# TerraScope — Satellite Change Intelligence Platform
## Research-Grade Multi-Architecture GPU Training & Benchmarking Pipeline

This notebook manages the reproducible training and quantitative benchmarking of three change detection architectures on Google Colab:
1. **Siamese U-Net**: Convolutional baseline with spatial skip-connections and difference fusion.
2. **Siamese Vision Transformer (ViT)**: Pure self-attention token-based feature extractor.
3. **Siamese Swin Transformer**: Shifted-window hierarchical vision transformer with multi-scale fusion.

### Benchmark Datasets:
- **LEVIR-CD**: Ultra-high-resolution bi-temporal optical building change detection (RGB).
- **OSCD**: Onera Satellite Change Detection multi-spectral Sentinel-2 dataset.

---
### Reliability & Resilience Safeguards:
- **Periodic Checkpoint Backup**: Model checkpoints (`latest.pt` and `best.pt`) are automatically synced to Google Drive **at the end of every epoch**, not just at completion.
- **Session Disconnect Recovery**: Automatic checkpoint resumption allows continuing from where training stopped if Colab times out.
- **Accidental Overwrite Protection**: Already completed experiments are safely skipped on re-runs unless explicitly requested via `FORCE_RETRAIN = True`.
- **Strict Verification Gates**: Dataset structure, label alignments, and a lightweight GPU smoke test must pass before long-running training begins.
- **Isolated Experiment Execution**: If an individual run encounters an issue, errors are captured and remaining experiments continue.


### Step 1: Mount Google Drive & Configure Persistent Storage
Mount Google Drive to access repository files, load datasets, and ensure checkpoints and logs survive Colab runtime recycling.


In [ ]:
from google.colab import drive
import os
import sys
import shutil
from pathlib import Path

drive.mount("/content/drive")

# Configure persistent output directories on Google Drive
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/TerraScope_Outputs")
DRIVE_CHECKPOINTS_DIR = DRIVE_OUTPUT_DIR / "checkpoints"
DRIVE_EXPERIMENTS_DIR = DRIVE_OUTPUT_DIR / "experiments"
DRIVE_VIZ_DIR = DRIVE_OUTPUT_DIR / "visualizations"

for p in (DRIVE_CHECKPOINTS_DIR, DRIVE_EXPERIMENTS_DIR, DRIVE_VIZ_DIR):
    p.mkdir(parents=True, exist_ok=True)

print(f" Persistent Google Drive storage initialized at: {DRIVE_OUTPUT_DIR}")


### Step 2: GPU Accelerator & CUDA Environment Audit
Verify that a CUDA-capable GPU is active. Raises a hard error if the runtime is set to CPU.


In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL: CUDA GPU is not available! \n"
        "Go to Runtime > Change runtime type > Hardware accelerator > select T4 GPU (or A100)."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f" PyTorch Version : {torch.__version__}")
print(f" CUDA Device     : {gpu_name}")
print(f" Total VRAM      : {total_vram_gb:.2f} GB")
if total_vram_gb < 11.0:
    print(f" Note: Total VRAM is {total_vram_gb:.1f} GB. Using batch size 8 is supported.")
print(" GPU accelerator passed hardware verification.")


### Step 3: Setup Project Workspace
Copies repository code to local `/content/satellite` for high-throughput SSD read/write during training, while keeping the persistent backup linked to Google Drive.


In [ ]:
DRIVE_PROJECT_PATH = Path("/content/drive/MyDrive/satellite")
LOCAL_PROJECT_PATH = Path("/content/satellite")

if DRIVE_PROJECT_PATH.exists():
    if not LOCAL_PROJECT_PATH.exists():
        print(f"Copying project from {DRIVE_PROJECT_PATH} to {LOCAL_PROJECT_PATH} for SSD throughput...")
        shutil.copytree(str(DRIVE_PROJECT_PATH), str(LOCAL_PROJECT_PATH), dirs_exist_ok=True)
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
elif LOCAL_PROJECT_PATH.exists():
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
else:
    PROJECT_ROOT = Path(os.getcwd())

print(f" Active Project Root: {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


### Step 4: Install Dependencies & Verify Core Modules
Install geospatial and vision transformer libraries.


In [ ]:
!pip install -q rasterio geopandas timm reportlab python-dotenv

# Verify crucial imports
import rasterio
import timm
from ml.models import get_model
from ml.utils.device import detect_device

print(f" Active Device Resolution: {detect_device()}")
print(" Core dependencies verified successfully.")


### Step 5: Dataset Path Configuration & Structural Integrity Gate
Configure dataset directories and validate their physical structure before training.
Execution stops with a clear error if required splits or labels are absent.


In [ ]:
# --- CONFIGURE YOUR DATASET PATHS ON GOOGLE DRIVE ---
LEVIR_CD_DIR = Path("/content/drive/MyDrive/data/LEVIR-CD")
OSCD_DIR = Path("/content/drive/MyDrive/data/onera DATA/Onera Satellite Change Detection dataset - Images")

def validate_datasets(levir_path: Path, oscd_path: Path) -> dict:
    status = {"levir_ok": False, "oscd_ok": False}
    
    # 1. Validate LEVIR-CD
    if not levir_path.exists():
        print(f" Note: LEVIR-CD path does not exist at {levir_path}")
    else:
        missing_levir = []
        for sp in ("train", "val", "test"):
            for sub in ("A", "B", "label"):
                d = levir_path / sp / sub
                if not d.is_dir() or len(list(d.glob("*.*"))) == 0:
                    missing_levir.append(str(d))
        if missing_levir:
            raise FileNotFoundError(
                f"LEVIR-CD missing required subdirectories/files:\n" + "\n".join(missing_levir)
            )
        status["levir_ok"] = True
        print(f" LEVIR-CD validated successfully: {levir_path}")

    # 2. Validate OSCD
    if not oscd_path.exists():
        print(f" Note: OSCD path does not exist at {oscd_path}")
    else:
        from ml.datasets.oscd import _find_images_base, _find_label_roots
        img_base = _find_images_base(oscd_path)
        train_lbl, test_lbl = _find_label_roots(oscd_path, img_base)
        if not train_lbl or not train_lbl.is_dir():
            raise FileNotFoundError(f"OSCD Train Labels folder not found near {oscd_path}")
        if not test_lbl or not test_lbl.is_dir():
            raise FileNotFoundError(f"OSCD Test Labels folder not found near {oscd_path}")
        status["oscd_ok"] = True
        print(f" OSCD validated successfully:")
        print(f"   Images Base : {img_base.name}")
        print(f"   Train Labels: {train_lbl.name}")
        print(f"   Test Labels : {test_lbl.name}")

    if not status["levir_ok"] and not status["oscd_ok"]:
        raise SystemExit("CRITICAL: Neither LEVIR-CD nor OSCD was found at the configured paths. Stopping.")
    return status

dataset_status = validate_datasets(LEVIR_CD_DIR, OSCD_DIR)


### Step 6: Lightweight Pre-Flight GPU Smoke Test
Executes a rapid 3-second in-memory test on CUDA:
1. CUDA forward pass
2. BCE + Dice loss computation
3. Backward gradient pass (`loss.backward()`)
4. Optimizer step (`optimizer.step()`)
5. Checkpoint serialization and reload integrity check

Execution immediately halts if any step fails, guaranteeing no wasted compute on a broken setup.


In [ ]:
import torch
import torch.nn as nn
from ml.models import get_model
from ml.training.losses import get_loss_function
from ml.training.optimizer import create_optimizer
from ml.training.callbacks import ModelCheckpointManager, load_checkpoint

print("=== STARTING LIGHTWEIGHT GPU SMOKE TEST ===")

try:
    device = torch.device("cuda")
    
    # 1. Instantiate small model on GPU
    smoke_model = get_model("siamese_unet", in_channels=3, img_size=256, pretrained=False).to(device)
    smoke_model.train()
    
    # 2. Synthetic bitemporal batch
    t1 = torch.randn(2, 3, 256, 256, device=device)
    t2 = torch.randn(2, 3, 256, 256, device=device)
    target = torch.randint(0, 2, (2, 256, 256), device=device, dtype=torch.float32)
    
    # 3. Forward pass
    logits = smoke_model(t1, t2)
    assert logits.shape == (2, 1, 256, 256), f"Unexpected output shape: {logits.shape}"
    
    # 4. Loss calculation
    criterion = get_loss_function("bce_dice")
    loss = criterion(logits, target)
    assert torch.isfinite(loss), f"Loss is not finite: {loss.item()}"
    
    # 5. Backward pass
    optimizer = create_optimizer(smoke_model, name="adamw", lr=3e-4)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 6. Checkpoint persistence check
    temp_ckpt_dir = Path("/tmp/smoke_checkpoints")
    ckpt_mgr = ModelCheckpointManager(temp_ckpt_dir, "smoke_unet", "test_ds")
    saved_path = ckpt_mgr.save(smoke_model, optimizer, None, epoch=1, best_metric=0.85, config={}, is_best=True)
    assert saved_path.is_file(), "Smoke checkpoint was not created!"
    
    # Reload test
    reload_meta = load_checkpoint(saved_path, smoke_model, optimizer, map_location=device)
    assert reload_meta.get("epoch") == 1, "Checkpoint reload failed!"
    
    print(f" Forward Pass: OK (Output shape: {logits.shape})")
    print(f" Loss Calc   : OK (Loss value: {loss.item():.4f})")
    print(f" Backward    : OK (Gradients computed)")
    print(f" Checkpoint  : OK (Saved & reloaded from {saved_path})")
    print("=== SMOKE TEST PASSED: GPU COMPUTE PIPELINE IS 100% OPERATIONAL ===")
except Exception as e:
    print(f"CRITICAL ERROR in Smoke Test: {e}")
    raise SystemExit("Smoke test failed! Halting notebook to prevent broken training runs.")


## Step 7: Systematic Model Training Pipeline

### Controlled Research Configuration:
Every architecture is trained with strictly identical optimization parameters:
- **Optimizer**: `AdamW` (`weight_decay=1e-4`)
- **Loss Function**: Combined Binary Cross-Entropy + Dice Loss (`bce_dice`)
- **Learning Rate**: $3 \times 10^{-4}$ with Cosine Annealing (`--scheduler cosine`)
- **Batch Size**: 8 (optimal for 15GB T4 GPU VRAM and gradient estimation stability)
- **Patch Resolution**: $256 \times 256$ pixels
- **Random Seed**: `42` across all Python, NumPy, and PyTorch RNGs

### Checkpoint Persistence Strategy:
- **Periodic Drive Sync**: `--backup-dir` is passed so `latest.pt` and `best.pt` are mirrored to Google Drive **every epoch**.
- **Accidental Overwrite Guard**: If an experiment is already marked `COMPLETED` and its checkpoint exists on Google Drive, it is **skipped** on notebook re-runs unless `FORCE_RETRAIN = True`.
- **Fault-Tolerant Loop**: If one experiment fails, its error is logged in `experiment_manifest.json` and subsequent experiments proceed.


In [ ]:
import subprocess
import json
import time

# Set to True only if you intentionally want to retrain already completed runs
FORCE_RETRAIN = False

EXPERIMENT_MATRIX = [
    # --- LEVIR-CD EXPERIMENTS ---
    {
        "id": "exp1_levir_unet",
        "model": "siamese_unet",
        "dataset": "LEVIR-CD",
        "data_dir": str(LEVIR_CD_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    {
        "id": "exp2_levir_vit",
        "model": "vit",
        "dataset": "LEVIR-CD",
        "data_dir": str(LEVIR_CD_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    {
        "id": "exp3_levir_swin",
        "model": "swin",
        "dataset": "LEVIR-CD",
        "data_dir": str(LEVIR_CD_DIR),
        "epochs": 20,
        "batch_size": 8,
        "bands": None,
        "enabled": dataset_status["levir_ok"],
    },
    # --- OSCD EXPERIMENTS ---
    {
        "id": "exp4_oscd_unet",
        "model": "siamese_unet",
        "dataset": "OSCD",
        "data_dir": str(OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
    {
        "id": "exp5_oscd_vit",
        "model": "vit",
        "dataset": "OSCD",
        "data_dir": str(OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
    {
        "id": "exp6_oscd_swin",
        "model": "swin",
        "dataset": "OSCD",
        "data_dir": str(OSCD_DIR),
        "epochs": 25,
        "batch_size": 8,
        "bands": ["B04", "B03", "B02"],
        "enabled": dataset_status["oscd_ok"],
    },
]

manifest_path = DRIVE_OUTPUT_DIR / "experiment_manifest.json"
manifest = {}
if manifest_path.exists():
    try:
        manifest = json.loads(manifest_path.read_text())
    except Exception:
        manifest = {}

print(f"Loaded existing manifest with {len(manifest)} recorded runs.")

for idx, exp in enumerate(EXPERIMENT_MATRIX, 1):
    exp_id = exp["id"]
    model_name = exp["model"]
    dataset_name = exp["dataset"]
    safe_ds = dataset_name.lower().replace(" ", "_")
    best_ckpt_drive = DRIVE_CHECKPOINTS_DIR / f"{model_name}_{safe_ds}_best.pt"
    latest_ckpt_drive = DRIVE_CHECKPOINTS_DIR / f"{model_name}_{safe_ds}_latest.pt"

    print(f"\n=======================================================")
    print(f"[{idx}/6] Experiment: {exp_id} ({model_name} on {dataset_name})")
    print(f"=======================================================")

    if not exp["enabled"]:
        print(f"  SKIPPED: Dataset for {dataset_name} is marked disabled/unavailable.")
        continue

    # Guard: Do not overwrite completed experiments
    if not FORCE_RETRAIN and best_ckpt_drive.exists() and manifest.get(exp_id, {}).get("status") == "COMPLETED":
        print(f"  SKIPPED: Experiment already completed with checkpoint: {best_ckpt_drive.name}")
        continue

    cmd = [
        sys.executable,
        "scripts/train_model.py",
        "--model", model_name,
        "--dataset", dataset_name,
        "--data-dir", exp["data_dir"],
        "--epochs", str(exp["epochs"]),
        "--batch-size", str(exp["batch_size"]),
        "--patch-size", "256",
        "--lr", "3e-4",
        "--loss", "bce_dice",
        "--optimizer", "adamw",
        "--scheduler", "cosine",
        "--device", "cuda",
        "--seed", "42",
        "--checkpoint-dir", "models/checkpoints",
        "--experiments-dir", "experiments",
        "--backup-dir", str(DRIVE_CHECKPOINTS_DIR),
    ]

    # Check for auto-resume if a latest checkpoint exists from an interrupted run
    if latest_ckpt_drive.exists():
        print(f"  Found previous run checkpoint! Resuming from: {latest_ckpt_drive.name}")
        cmd.extend(["--resume", str(latest_ckpt_drive)])

    if exp["bands"]:
        cmd.append("--bands")
        cmd.extend(exp["bands"])

    print(f"  Launching training command: {' '.join(cmd[:6])} ...")
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, check=True)
        duration_sec = round(time.time() - t0, 2)
        local_best = Path("models/checkpoints") / f"{model_name}_{safe_ds}_best.pt"
        
        if local_best.exists():
            shutil.copy2(local_best, best_ckpt_drive)
            manifest[exp_id] = {
                "status": "COMPLETED",
                "model": model_name,
                "dataset": dataset_name,
                "checkpoint": str(best_ckpt_drive),
                "duration_sec": duration_sec,
                "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            print(f"  Experiment {exp_id} COMPLETED in {duration_sec}s. Checkpoint synced.")
        else:
            manifest[exp_id] = {"status": "FAILED", "error": "Best checkpoint not found after training"}
            print(f"  WARNING: Process exited cleanly but {local_best.name} was not created.")
    except subprocess.CalledProcessError as err:
        duration_sec = round(time.time() - t0, 2)
        manifest[exp_id] = {
            "status": "FAILED",
            "error": f"Process exited with code {err.returncode}",
            "duration_sec": duration_sec,
        }
        print(f"  ERROR: Experiment {exp_id} failed with exit code {err.returncode}. Continuing remaining runs.")
    except Exception as exc:
        manifest[exp_id] = {"status": "FAILED", "error": str(exc)}
        print(f"  ERROR: Unexpected failure in {exp_id}: {exc}. Continuing remaining runs.")

    # Save manifest snapshot to Google Drive immediately after each run
    manifest_path.write_text(json.dumps(manifest, indent=2))

print("\n Training sequence finished. Manifest saved to Google Drive.")


## Step 8: Standardized Quantitative Benchmarking on Test Split
Evaluates the three trained models under strictly identical test conditions:
- **Metrics**: Precision, Recall, F1-Score, IoU, Overall Accuracy, Kappa
- **Visuals**: Pixel-level Confusion Matrices & Qualitative Prediction Overlays
- **Reports**: JSON, CSV, and Markdown saved to `experiments/results/` and synced to Drive


In [ ]:
if dataset_status["levir_ok"]:
    print("=== Running LEVIR-CD Standardized Benchmark Evaluation ===")
    subprocess.run([
        sys.executable,
        "scripts/evaluate_model.py",
        "--dataset", "LEVIR-CD",
        "--data-dir", str(LEVIR_CD_DIR),
        "--split", "test",
        "--patch-size", "256",
        "--batch-size", "8",
        "--threshold", "0.5",
        "--device", "cuda",
        "--checkpoint-dir", str(DRIVE_CHECKPOINTS_DIR),
        "--experiments-dir", str(DRIVE_EXPERIMENTS_DIR),
        "--save-visuals",
        "--visual-samples", "6",
    ], check=True)


In [ ]:
if dataset_status["oscd_ok"]:
    print("=== Running OSCD Standardized Benchmark Evaluation ===")
    subprocess.run([
        sys.executable,
        "scripts/evaluate_model.py",
        "--dataset", "OSCD",
        "--data-dir", str(OSCD_DIR),
        "--split", "test",
        "--patch-size", "256",
        "--batch-size", "8",
        "--threshold", "0.5",
        "--device", "cuda",
        "--bands", "B04", "B03", "B02",
        "--checkpoint-dir", str(DRIVE_CHECKPOINTS_DIR),
        "--experiments-dir", str(DRIVE_EXPERIMENTS_DIR),
        "--save-visuals",
        "--visual-samples", "6",
    ], check=True)


## Step 9: Render Model Comparison Charts & Qualitative Visuals


In [ ]:
import glob
from IPython.display import Image, display

# 1. Comparison bar charts
charts = sorted(glob.glob("outputs/visualizations/*_comparison.png"))
if charts:
    print(f"Found {len(charts)} comparison charts:")
    for c in charts:
        print(f"Displaying: {c}")
        display(Image(filename=c))

# 2. Confusion Matrices
cms = sorted(glob.glob("outputs/visualizations/*_cm.png"))
if cms:
    print(f"\nFound {len(cms)} confusion matrices:")
    for cm in cms:
        display(Image(filename=cm))


## Step 10: Final Google Drive Backup Audit & Verification
Synchronizes any remaining outputs and verifies file existence and non-zero sizes in persistent Google Drive storage.


In [ ]:
import os
import time

print("=== Synchronizing all final outputs to Google Drive ===")
# Sync outputs to Drive
!cp -r outputs/visualizations/* "$DRIVE_VIZ_DIR/" 2>/dev/null || true
!cp -r experiments/results/* "$DRIVE_EXPERIMENTS_DIR/" 2>/dev/null || true

print("\n=== AUDIT: Verified Files in Google Drive Storage ===")

def audit_dir(name: str, path: Path):
    print(f"\n[{name}] -> {path}")
    if not path.is_dir():
        print("  Directory does not exist!")
        return
    files = sorted(path.glob("*.*"))
    if not files:
        print("  (Empty directory)")
        return
    for f in files:
        size_mb = f.stat().st_size / (1024 * 1024)
        mtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(f.stat().st_mtime))
        print(f"   {f.name:<45} | {size_mb:>8.2f} MB | {mtime}")

audit_dir("Checkpoints", DRIVE_CHECKPOINTS_DIR)
audit_dir("Experiments & Benchmarks", DRIVE_EXPERIMENTS_DIR)
audit_dir("Visualizations", DRIVE_VIZ_DIR)

print("\n Audit completed. All experimental assets are safely preserved on Google Drive.")
